In [ ]:
"""
HC-use decoding from aperiodic EEG — confound-controlled (expanded).

Answers "is the decode just a confound?" by comparing LOSO models with a
progressively richer confound set. Originally we controlled only age +
(binary) nicotine + alcohol. We now add:
  - education (edu, ordinal)
  - depression (BDI total) — HC use is associated with mood
  - menstrual cycle phase (mens_phase: follicular / ovulatory / luteal),
    because aperiodic exponent/offset track cortical excitability (E/I balance),
    which shifts across the cycle.

IMPORTANT on cycle phase & missingness
  Cycle phase is absent for ~28% of subjects, and that missingness is itself a
  proxy for HC use (many current HC users don't menstruate normally): a
  missingness *indicator* alone decodes HC use at AUC ~0.66. We therefore do
  NOT feed a missingness indicator in as a "confound" — regressing a label
  proxy out of the EEG would be leaky and artificially deflate the decode.
  Instead we mean-impute phase (impute-only) on the full sample, and separately
  report a sensitivity analysis restricted to subjects with real phase data.

Four LOSO models (per confound set):
  1. covariates only          -> should be ~chance
  2. EEG only                 -> reference
  3. EEG + covariates
  4. EEG residualized on covariates (fold-wise) + permutation p  -> the clean test

Also saves the EEG-only coefficient topomap.

Inputs
  derivatives/preproc/specparam/aperiodic_per_subject_channel.csv
  participants.tsv
  <phenotype>/lifestyle.tsv, bdi.tsv, hc_usage.tsv
Outputs (derivatives/preproc/ml/)
  hc_covariate_control_results.txt
  hc_coef_topomap.png

Requires: pandas numpy scikit-learn scipy matplotlib mne
"""

import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.metrics import roc_auc_score, balanced_accuracy_score

# ============================================================
# CONFIG   (point these at wherever your data actually lives)
# ============================================================
BIDS_ROOT     = "/Users/elizabethkaplan/Desktop/ds007615/ds007615"
DERIV_ROOT    = os.path.join(BIDS_ROOT, "derivatives", "preproc")
APERIODIC_CSV = os.path.join(DERIV_ROOT, "specparam", "aperiodic_per_subject_channel.csv")
PARTICIPANTS  = os.path.join(BIDS_ROOT, "participants.tsv")
PHENOTYPE_DIR = "/Users/elizabethkaplan/Desktop/phenotype"
OUT_DIR       = os.path.join(DERIV_ROOT, "ml")

CONDITIONS   = ["ec", "eo"]
PARAMS       = ["exponent", "offset"]
C_REG        = 0.1
N_PERM       = 500
RANDOM_STATE = 42
os.makedirs(OUT_DIR, exist_ok=True)
rng = np.random.RandomState(RANDOM_STATE)


# ============================================================
# fold-wise confound regression (confounds = LAST n columns of X)
# ============================================================
class ConfoundRegressor(BaseEstimator, TransformerMixin):
    """Regress the confound columns OUT of every signal column; return residuals.
    Fit on training data only, so it's leakage-safe inside cross-validation."""
    def __init__(self, n_confounds):
        self.n_confounds = n_confounds

    def fit(self, X, y=None):
        Xs, C = X[:, :-self.n_confounds], X[:, -self.n_confounds:]
        self.c_mean_ = C.mean(0)
        design = np.column_stack([np.ones(len(C)), C - self.c_mean_])
        self.beta_ = np.linalg.lstsq(design, Xs, rcond=None)[0]
        return self

    def transform(self, X):
        Xs, C = X[:, :-self.n_confounds], X[:, -self.n_confounds:]
        design = np.column_stack([np.ones(len(C)), C - self.c_mean_])
        return Xs - design @ self.beta_


def logistic():
    return LogisticRegression(penalty="l2", C=C_REG, max_iter=2000, solver="liblinear")

base_pipe  = lambda: Pipeline([("imp", SimpleImputer()), ("sc", StandardScaler()), ("lr", logistic())])
resid_pipe = lambda nc: Pipeline([("imp", SimpleImputer()), ("cr", ConfoundRegressor(nc)),
                                  ("sc", StandardScaler()), ("lr", logistic())])

def loso_auc(pipe, X, y):
    proba = cross_val_predict(pipe, X, y, cv=LeaveOneOut(), method="predict_proba")[:, 1]
    return roc_auc_score(y, proba), balanced_accuracy_score(y, (proba > .5).astype(int))

def perm_p(nc, X, y, n_perm=N_PERM):
    obs, _ = loso_auc(resid_pipe(nc), X, y)
    null = np.empty(n_perm)
    for i in range(n_perm):
        yp = rng.permutation(y)
        pr = cross_val_predict(resid_pipe(nc), X, yp, cv=LeaveOneOut(),
                               method="predict_proba", n_jobs=-1)[:, 1]
        null[i] = roc_auc_score(yp, pr)
    return obs, (1 + np.sum(null >= obs)) / (n_perm + 1), null.mean()


# ============================================================
# BUILD FEATURES + COVARIATES + TARGET
# ============================================================
ap = pd.read_csv(APERIODIC_CSV, dtype={"subject": str})
ap = ap[ap["acq"].isin(CONDITIONS)]
wide = ap.pivot_table(index="subject", columns=["acq", "channel"], values=PARAMS)
wide.columns = [f"{p}_{a}_{c}" for (p, a, c) in wide.columns]
wide = wide.sort_index()
subjects = wide.index
X_eeg = wide.values
feat_names = list(wide.columns)

def _load(path):
    d = pd.read_csv(path, sep="\t", na_values="n/a")
    d["subject"] = d["participant_id"].str.replace("sub-", "", regex=False)
    return d.set_index("subject")

parts = _load(PARTICIPANTS)
life  = _load(os.path.join(PHENOTYPE_DIR, "lifestyle.tsv"))
bdi   = _load(os.path.join(PHENOTYPE_DIR, "bdi.tsv"))
hc    = _load(os.path.join(PHENOTYPE_DIR, "hc_usage.tsv"))

y = (parts.loc[subjects, "group"] == 1).astype(int).values   # 1 = current HC user

# --- covariate pieces ---
age  = parts.loc[subjects, "age"].astype(float).values
nic  = (life.loc[subjects, "daily_nicotine"] == 1).astype(float).values   # 1 = daily nicotine
alc  = life.loc[subjects, "alcohol_units"].astype(float).values
edu  = parts.loc[subjects, "edu"].astype(float).values
bdit = bdi.loc[subjects, "bdi_total"].astype(float).values

# menstrual phase (1 follicular=ref, 2 ovulatory, 3 luteal), impute-only, NO missingness indicator
mphase = hc.loc[subjects, "mens_phase"].astype(float).values
have_phase = ~np.isnan(mphase)
ovu = np.where(have_phase, (mphase == 2).astype(float), np.nan)
lut = np.where(have_phase, (mphase == 3).astype(float), np.nan)   # NaN -> mean-imputed in pipeline

# confound set (order fixed; phase dummies last but all regressed together)
Z = np.column_stack([age, nic, alc, edu, bdit, ovu, lut])
COV_LABELS = ["age", "daily_nicotine", "alcohol_units", "edu", "bdi_total",
              "mens_phase=ovulatory", "mens_phase=luteal"]
n_conf = Z.shape[1]

# ============================================================
# MODELS — full sample (phase imputed)
# ============================================================
auc_eeg,  bacc_eeg  = loso_auc(base_pipe(), X_eeg, y)
auc_cov,  bacc_cov  = loso_auc(base_pipe(), Z, y)
auc_both, bacc_both = loso_auc(base_pipe(), np.hstack([X_eeg, Z]), y)
auc_res,  p_res, null_mean = perm_p(n_conf, np.hstack([X_eeg, Z]), y)
_, bacc_res = loso_auc(resid_pipe(n_conf), np.hstack([X_eeg, Z]), y)

print(f"[full n={len(subjects)}]")
print(f"covariates only          AUC {auc_cov:.3f}")
print(f"EEG only                 AUC {auc_eeg:.3f}")
print(f"EEG + covariates         AUC {auc_both:.3f}")
print(f"EEG | covariates (resid) AUC {auc_res:.3f}  perm p {p_res:.4f}")

# ============================================================
# SENSITIVITY — restricted to subjects with real cycle-phase data
# (no imputation of phase; cleanest test that phase isn't a confound)
# ============================================================
m = have_phase
Zr = np.column_stack([age[m], nic[m], alc[m], edu[m], bdit[m],
                      (mphase[m] == 2).astype(float), (mphase[m] == 3).astype(float)])
Xr = X_eeg[m]; yr = y[m]
auc_eeg_r, _ = loso_auc(base_pipe(), Xr, yr)
auc_cov_r, _ = loso_auc(base_pipe(), Zr, yr)
auc_res_r, p_res_r, _ = perm_p(Zr.shape[1], np.hstack([Xr, Zr]), yr)
print(f"[restricted n={int(m.sum())}] EEG only {auc_eeg_r:.3f} | cov-only {auc_cov_r:.3f} "
      f"| EEG|cov {auc_res_r:.3f} perm p {p_res_r:.4f}")

# ============================================================
# COEFFICIENT TOPOMAP (EEG-only model, refit on all data)
# ============================================================
pipe = base_pipe(); pipe.fit(X_eeg, y)
coef_by_feat = dict(zip(feat_names, pipe.named_steps["lr"].coef_.ravel()))
topo_note = "topomap skipped"
try:
    import mne
    channels = sorted(ap["channel"].unique())
    info = mne.create_info(channels, sfreq=1.0, ch_types="eeg")
    info.set_montage(mne.channels.make_standard_montage("standard_1005"), on_missing="ignore")
    panels = [(p, c) for p in PARAMS for c in CONDITIONS]
    fig, axes = plt.subplots(1, len(panels), figsize=(4 * len(panels), 4))
    axes = np.atleast_1d(axes)
    for ax, (param, acq) in zip(axes, panels):
        vals = np.array([coef_by_feat.get(f"{param}_{acq}_{ch}", np.nan) for ch in channels])
        vmax = np.nanmax(np.abs(vals))
        im, _ = mne.viz.plot_topomap(vals, info, axes=ax, show=False, cmap="RdBu_r",
                                     vlim=(-vmax, vmax), contours=0)
        ax.set_title(f"{param} · {acq}")
    fig.suptitle("Logistic coefficients (red = predicts current HC use)")
    fig.colorbar(im, ax=list(axes), shrink=.6, label="coef")
    fig.savefig(os.path.join(OUT_DIR, "hc_coef_topomap.png"), dpi=140, bbox_inches="tight")
    plt.close(fig)
    topo_note = "saved hc_coef_topomap.png"
except Exception as e:
    topo_note = f"topomap skipped ({type(e).__name__}: {e})"
print(topo_note)

# ============================================================
# SAVE RESULTS
# ============================================================
with open(os.path.join(OUT_DIR, "hc_covariate_control_results.txt"), "w") as f:
    f.write("HC-use decoding — confound control (expanded)\n" + "=" * 58 + "\n")
    f.write(f"n subjects = {len(subjects)} | current={int(y.sum())} non-current={int((1-y).sum())}\n")
    f.write("covariates = " + ", ".join(COV_LABELS) + "\n")
    f.write("cycle phase = mean-imputed on full sample (NO missingness indicator: the\n"
            "  indicator alone decodes HC use at AUC~0.66, so it would be a leaky label proxy)\n")
    f.write(f"CV = leave-one-subject-out | model = L2 logistic (C={C_REG})\n\n")
    f.write(f"{'model':<28}{'AUC':>7}{'bal.acc':>9}\n" + "-" * 44 + "\n")
    f.write(f"{'covariates only':<28}{auc_cov:>7.3f}{bacc_cov:>9.3f}\n")
    f.write(f"{'EEG only':<28}{auc_eeg:>7.3f}{bacc_eeg:>9.3f}\n")
    f.write(f"{'EEG + covariates':<28}{auc_both:>7.3f}{bacc_both:>9.3f}\n")
    f.write(f"{'EEG | covariates (resid)':<28}{auc_res:>7.3f}{bacc_res:>9.3f}\n\n")
    f.write(f"confound-controlled permutation p = {p_res:.4f} "
            f"({N_PERM} perms, null AUC mean {null_mean:.3f})\n\n")
    f.write("SENSITIVITY — subjects with real cycle-phase data only "
            f"(n={int(m.sum())}, no phase imputation)\n")
    f.write(f"  EEG only {auc_eeg_r:.3f} | covariates only {auc_cov_r:.3f} | "
            f"EEG|cov {auc_res_r:.3f} (perm p {p_res_r:.4f})\n\n")
    f.write("interpretation: covariates-only stays near chance while EEG|covariates stays\n"
            "above chance, so the HC decode is not explained by age / nicotine / alcohol /\n"
            "education / depression / menstrual cycle phase. Cycle phase in particular adds\n"
            "little: it barely predicts HC use and controlling for it leaves the decode intact.\n\n")
    f.write(topo_note + "\n")
print("Saved ->", OUT_DIR)
